## استيراد المكتبات

هنا بس نستورد المكتبات اللي بنستخدمها في المشروع.
csv عشان نقرأ الملف،
numpy عشان نحسب الـ cosine similarity،
pickle عشان نحفظ النتيجة،
و string عشان نساعد في حذف علامات الترقيم.

In [1]:
import csv
import numpy as np
import pickle
import string

## قراءة ملف المقالات

هنا نقرأ ملف articles.csv
ونخزن كل مقال داخل ليست اسمها articles
كل مقال يكون عبارة عن dictionary فيه id و title و content

In [4]:
articles = []
with open("articles.csv", "r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        articles.append(row)

len(articles)

5

## تنظيف النص

الآن ننظف محتوى كل مقال:
نحول الكلام لحروف صغيرة،
نشيل علامات الترقيم،
ونشيل الأرقام.
وبعدها نحفظ النص النظيف داخل المقال.

In [5]:
def clean_text(text):
    text = text.lower()
    text = "".join([c for c in text if c not in string.punctuation])
    text = "".join([c for c in text if not c.isdigit()])
    return text

for article in articles:
    article["clean_content"] = clean_text(article["content"])

articles[0]["clean_content"][:200]

'artificial intelligence is transforming industries globally machine learning and deep learning are key components of modern ai systems'

## بناء الـ Bag of Words

هنا نجمع كل الكلمات من كل المقالات
ونسوي ليست فيها الكلمات بدون تكرار.
هذه هي الـ vocabulary اللي بنعتمد عليها في بناء الـ vectors.

In [6]:
vocab = []
for article in articles:
    words = article["clean_content"].split()
    for word in words:
        if word not in vocab:
            vocab.append(word)

len(vocab)

58

## تحويل النص إلى Vector

الدالة هذه تأخذ نص + vocabulary
وترجع متجه:
لو الكلمة موجودة نحط عدد تكرارها
لو مش موجودة نحط صفر.

In [7]:
def text_to_vector(text, bag_of_words):

    words = text.lower().split()
    
    vector = []
    
    for word in bag_of_words:
        if word in words:
            vector.append(words.count(word))
        else:
            vector.append(0)
    
    return vector

## بناء المتجهات لكل المقالات

الآن نطبق الدالة على كل مقال
ونحول الناتج إلى numpy array
عشان نقدر نحسب الـ cosine similarity.

In [8]:
vectors = []
for article in articles:
    vec = text_to_vector(article["clean_content"], vocab)
    vectors.append(np.array(vec))

vectors[0][:20]

array([1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0])

## حساب Cosine Similarity

هنا نحسب التشابه بين كل مقال وكل مقال ثاني
باستخدام:
dot product
و norm
وبعدين نخزن النتيجة في مصفوفة مربعة.

In [9]:
n = len(vectors)
similarity_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        dot = np.dot(vectors[i], vectors[j])
        norm_i = np.linalg.norm(vectors[i])
        norm_j = np.linalg.norm(vectors[j])

        if norm_i == 0 or norm_j == 0:
            similarity = 0
        else:
            similarity = dot / (norm_i * norm_j)

        similarity_matrix[i][j] = similarity

similarity_matrix

array([[1.        , 0.35      , 0.08944272, 0.5       , 0.09325048],
       [0.35      , 1.        , 0.04472136, 0.25      , 0.13987572],
       [0.08944272, 0.04472136, 1.        , 0.        , 0.50043459],
       [0.5       , 0.25      , 0.        , 1.        , 0.13987572],
       [0.09325048, 0.13987572, 0.50043459, 0.13987572, 1.        ]])

## حفظ النتيجة

نحفظ مصفوفة التشابه في ملف اسمه similarities.pkl
وهذا هو الناتج الأساسي المطلوب في التكليف.

In [10]:
with open("similarities.pkl", "wb") as f:
    pickle.dump(similarity_matrix, f)

"saved similarities.pkl"

'saved similarities.pkl'

## إيجاد أكثر 3 مقالات تشابه

الدالة هذه تأخذ رقم مقال
وترجع أعلى 3 مقالات مشابهة له
بدون ما تحسب المقال نفسه.

In [11]:
def get_top3_similar(article_id):
    index = None
    for i in range(len(articles)):
        if articles[i]["id"] == str(article_id):
            index = i
            break

    if index is None:
        return []

    sims = similarity_matrix[index]
    pairs = []

    for i in range(len(sims)):
        if i != index:
            pairs.append((i, sims[i]))

    pairs.sort(key=lambda x: x[1], reverse=True)

    top3 = []
    for i in range(min(3, len(pairs))):
        top3.append(articles[pairs[i][0]]["title"])

    return top3

## تجربة سريعة

نجرب الدالة ونعطيها رقم مقال.

In [12]:
get_top3_similar(1)

['Machine Learning Trends', 'Future of Robotics', 'Big Data Analytics']